In [ ]:
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
import scipy.interpolate as spi
from scipy.io import savemat
import time

import desc
!pip show desc-opt

### output2D：二维平衡输出

本 notebook 读取 DESC 生成的二维平衡 `.hdf5` 文件，并输出 MATLAB 可读取的二维平衡数据。

最终将在目标文件夹中生成 `standard2D.mat`、`refined2D.mat` 和 `plot2D.mat`。

请按步骤运行需要的代码块；每一步运行前先确认相应代码块参数。

### 第一步：设置输入和输出路径

在 `1.1代码块` 中设置 `inputPath` 和 `outputPath`。

`inputPath` 是已经计算好的 DESC 平衡 `.hdf5` 文件路径；`outputPath` 是 `.mat` 文件的输出文件夹。运行后会自动创建输出文件夹。

In [ ]:
################## 1.1代码块 ##################

from pathlib import Path

inputPath = r'/mnt/c/Users/Desktop/EQ.hdf5'
outputPath = r'/mnt/c/Users//Desktop/EQ/'

folder_path = Path(outputPath)
folder_path.mkdir(exist_ok=True)

### 第二步：定义输出函数

运行 `2.1代码块` 定义后续输出所需函数。本步骤不会立即生成文件。

In [ ]:
################## 2.1代码块 ##################

def getRhoThetaPEST(staggered,rho_num,rho_min,rho_max,theta_num):
    
    rho_grid = np.linspace(rho_min, rho_max, rho_num)

    #设定我们想要的theta_pest网格
    
    if(staggered==0):
        dtheta_pest = 2*np.pi/theta_num
        theta_pest_min = -np.pi + dtheta_pest/2
        theta_pest_max =  np.pi - dtheta_pest/2
        theta_pest = np.linspace(theta_pest_min, theta_pest_max,theta_num)
    if(staggered==1):
        dtheta_pest = 2*np.pi/theta_num
        theta_pest_min = -np.pi
        theta_pest_max =  np.pi - dtheta_pest
        theta_pest = np.linspace(theta_pest_min, theta_pest_max,theta_num)

            
    return rho_grid, theta_pest
    

############################################################################################################
############################################################################################################

def outputStandard2D(staggered, rho_grid, theta_pest, n_segments):
    
    rho_num = rho_grid.size
    theta_num = theta_pest.size
    theta_desc = np.zeros((rho_num, theta_num))

    # DESC坐标逆变度规及其导数全，协变度规全但其导数不全
    
    matrix_list = \
    ['R','R_r','R_t',\
     'Z','Z_r','Z_t',\
     'sqrt(g)',\
     '|B|','|B|_r','|B|_t','|B|_rr','|B|_rt','|B|_tt',\
     'lambda','lambda_r','lambda_t','lambda_rr','lambda_rt','lambda_tt',\
     'g^rr','g^rt','g^rz',\
     'g^tt','g^tz','g^zz',\
     'g^rr_r','g^rr_t',\
     'g^rt_r','g^rt_t',\
     'g^rz_r','g^rz_t',\
     'g^tt_r','g^tt_t',\
     'g^tz_r','g^tz_t',\
     'g^zz_r','g^zz_t',\
     'g_rr','g_rt','g_rz','g_tt','g_tt_r','g_tz','g_tz_r','g_zz',\
     'sqrt(g)_PEST','(sqrt(g)_PEST_r)|PEST','(sqrt(g)_PEST_v)|PEST',\
     'g^rr','g^rv','g^rz',\
     'g^zz',\
     'g_rr|PEST','g_rv|PEST','g_rp|PEST',\
     'g_vv|PEST','g_vp|PEST','g_pp|PEST',\
     '(g^rr_v)|PEST','(g^rv_v)|PEST','(g^rz_v)|PEST',\
     '(g_rr_v)|PEST','(g_vv_r)|PEST','(g_pp_v)|PEST',\
     'rho','q','iota','iota_r','iota_rr','e^vartheta','e^zeta']
    
    matrix_num = len(matrix_list)
    output = np.zeros((matrix_num, rho_num, theta_num))
    compute_list = matrix_list + ['theta']
    
    # 分段处理
    segment_size = rho_num // n_segments  # 假设整除
    # 用于收集 e^vartheta 和 e^zeta 的数据
    e_vartheta_all = np.zeros((rho_num * theta_num, 3))
    e_zeta_all = np.zeros((rho_num * theta_num, 3))
    
    for seg in range(n_segments):
        
        print(seg)
        
        start = seg * segment_size
        end = (seg + 1) * segment_size
        
        rho_seg = rho_grid[start:end]
        
        # 构建当前段的网格点
        grid_index = np.zeros((segment_size * theta_num, 3))
        for i in range(segment_size):
            for j in range(theta_num):
                grid_index[i * theta_num + j, 0] = rho_seg[i]
                grid_index[i * theta_num + j, 1] = theta_pest[j]
        
        field = eq.compute(compute_list, desc.grid.Grid(grid_index, coordinates='rvp'))
        
        # 填充前 matrix_num-2 个物理量（即除 e^vartheta 和 e^zeta 之外的所有量）
        for k in range(matrix_num - 2):
            temp = np.array(field[matrix_list[k]])
            for i in range(segment_size):
                for j in range(theta_num):
                    output[k, start + i, j] = temp[i * theta_num + j]
        
        theta_temp = np.array(field['theta'])
        for i in range(segment_size):
            for j in range(theta_num):
                theta_desc[start + i, j] = theta_temp[i * theta_num + j]
        
        # 收集 e^vartheta 和 e^zeta
        ev = field['e^vartheta']
        ez = field['e^zeta']
        for i in range(segment_size):
            for j in range(theta_num):
                idx = (start + i) * theta_num + j
                e_vartheta_all[idx, :] = ev[i * theta_num + j, :]
                e_zeta_all[idx, :] = ez[i * theta_num + j, :]
    
    # 计算 PESTcon_yy 和 PESTcon_yz
    PESTcon_yy = np.sum(e_vartheta_all * e_vartheta_all, axis=1)
    PESTcon_yz = np.sum(e_vartheta_all * e_zeta_all, axis=1)
    
    # 重新整理为 (rho_num, theta_num) 形状
    PESTcon_yy = PESTcon_yy.reshape(rho_num, theta_num)
    PESTcon_yz = PESTcon_yz.reshape(rho_num, theta_num)
    
    # 存入 output[69] 和 output[70]（原代码中的预留位置）
    output[69] = PESTcon_yy
    output[70] = PESTcon_yz
    
    # 构造 theta_p 数组（theta_pest 沿径向重复）
    theta_p = np.zeros((rho_num, theta_num))
    for j in range(theta_num):
        theta_p[:, j] = theta_pest[j]
    
    if staggered == 0:
        ifStaggered = 0
    else:
        ifStaggered = 1
    
    B00 = eq.compute('|B|', desc.grid.Grid(np.array([0, 0, 0])))['|B|']
    L00 = eq.compute('R', desc.grid.Grid(np.array([0, 0, 0])))['R']
    psit_max = eq.compute('psi', desc.grid.Grid(np.array([1, 0, 0])))['psi']

    iotaOrders = eq.iota.basis.modes[:,0]
    iotaCoeffs = eq.iota.params

    psipOrders = iotaOrders + 2;
    psipCoeffs = iotaCoeffs / psipOrders

    dpsipdrOrders = iotaOrders + 1;
    dpsipdrCoeffs = iotaCoeffs;

    psipCoeffs = psipCoeffs * 2 * psit_max
    dpsipdrCoeffs = dpsipdrCoeffs * 2 * psit_max

    psipPoly = desc.profiles.PowerSeriesProfile(params=psipCoeffs, modes=psipOrders)
    dpsipdrPoly = desc.profiles.PowerSeriesProfile(params=dpsipdrCoeffs, modes=dpsipdrOrders)

    psipIndex = np.zeros((rho_grid.size,3))
    psipIndex[:,0] = rho_grid

    psip = np.array(psipPoly.compute(desc.grid.Grid(psipIndex)));
    dpsipdr = np.array(dpsipdrPoly.compute(desc.grid.Grid(psipIndex)));

    psip = np.repeat(psip[:, np.newaxis], theta_num, axis=1)
    dpsipdr = np.repeat(dpsipdr[:, np.newaxis], theta_num, axis=1)
    
    path = outputPath + '/standard2D.mat'
    
    savemat(path,
            {'R': output[0], 'dR_dr': output[1], 'dR_dt': output[2],
             'Z': output[3], 'dZ_dr': output[4], 'dZ_dt': output[5],
             'JDESC': output[6],
             'B': output[7], 'dB_dr': output[8], 'dB_dt': output[9], 'dB_dr2': output[10], 'dB_drdt': output[11], 'dB_dt2': output[12],
             'lambda': output[13], 'dlambda_dr': output[14], 'dlambda_dt': output[15],
             'dlambda_dr2': output[16], 'dlambda_drdt': output[17], 'dlambda_dt2': output[18],
             'gcon_rr': output[19], 'gcon_rt': output[20], 'gcon_rz': output[21],
             'gcon_tt': output[22], 'gcon_tz': output[23], 'gcon_zz': output[24],
             'dgcon_rr_dr': output[25], 'dgcon_rr_dt': output[26],
             'dgcon_rt_dr': output[27], 'dgcon_rt_dt': output[28],
             'dgcon_rz_dr': output[29], 'dgcon_rz_dt': output[30],
             'dgcon_tt_dr': output[31], 'dgcon_tt_dt': output[32],
             'dgcon_tz_dr': output[33], 'dgcon_tz_dt': output[34],
             'dgcon_zz_dr': output[35], 'dgcon_zz_dt': output[36],
             'gcov_rr': output[37], 'gcov_rt': output[38], 'gcov_rz': output[39],
             'gcov_tt': output[40], 'gcov_tz': output[42], 'gcov_zz': output[44],
             'JPEST': output[45], 'dJPEST_drho': output[46], 'dJPEST_dtheta': output[47],
             'PESTcon_xx': output[48], 'PESTcon_xy': output[49], 'PESTcon_xz': output[50],
             'PESTcon_zz': output[51], 'PESTcon_yy': output[69], 'PESTcon_yz': output[70],
             'PESTcov_xx': output[52], 'PESTcov_xy': output[53], 'PESTcov_xz': output[54],
             'PESTcov_yy': output[55], 'PESTcov_yz': output[56], 'PESTcov_zz': output[57],
             'dPESTcon_xx_py': output[58], 'dPESTcon_xy_py': output[59], 'dPESTcon_xz_py': output[60],
             'dPESTcov_xx_py': output[61], 'dPESTcov_yy_px': output[62], 'dPESTcov_zz_py': output[63],
             'rho': output[64], 'q': output[65], 'iota': output[66], 'diota_dr': output[67], 'diota_dr2': output[68],
             'theta_pest': theta_p, 'theta_desc': theta_desc, 'ifStaggered': ifStaggered, 'B00': B00, 'L00': L00, 'psit_max': psit_max,
             'psip': psip, 'dpsip_dr': dpsipdr})

    
############################################################################################################
############################################################################################################


def outputRefined2D(refined_rho_0, refined_rho_1, refined_rho_num, refined_theta_num, n_segments):
    
    matrix_list = ['|B|','g_rr','g_rt','g_rz','g_tt','g_tz','g_zz','J_parallel','rho','theta']
    
    matrix_num=len(matrix_list)
    output=np.zeros((matrix_num,refined_rho_num,refined_theta_num))
    
    rho_grid = np.linspace(refined_rho_0,refined_rho_1,refined_rho_num)
    rho_grids = np.split(rho_grid, n_segments)
    segment_size = int(refined_rho_num / n_segments)
    
    theta_desc = np.linspace(-np.pi,np.pi,refined_theta_num + 1)
    theta_desc = theta_desc[0:refined_theta_num]

    grid_index = np.zeros((segment_size * refined_theta_num, 3))
    
    for index in range(0, n_segments):
        
        print(index)
        
        for i in range(0, segment_size):
            for j in range(0, refined_theta_num):
                grid_index[i*refined_theta_num + j][0] = rho_grids[index][i]
                grid_index[i*refined_theta_num + j][1] = theta_desc[j]
                
        field = eq.compute(matrix_list, desc.grid.Grid(grid_index, coordinates='rtz'))
        
        for k in range(0, matrix_num):
            temp = np.array(field[matrix_list[k]])
            for i in range(0, segment_size):
                for j in range(0, refined_theta_num):
                    output[k][i+index*segment_size][j] = temp[i*refined_theta_num + j]
    
    path = outputPath + '/refined2D.mat'
    
    savemat(path,\
            {'refined_B':output[0],\
             'refined_gcov_rr':output[1], 'refined_gcov_rt':output[2], 'refined_gcov_rz':output[3],\
             'refined_gcov_tt':output[4], 'refined_gcov_tz':output[5], 'refined_gcov_zz':output[6],\
             'refined_Jpara':output[7], 'refined_rho':output[8], 'refined_theta':output[9]})


############################################################################################################
############################################################################################################


def outputPlot2D(rho_grid, theta_pest, n_segments):
    
    rho_num = rho_grid.size
    theta_num = theta_pest.size
    
    matrix_list = ['q','R','Z','rho']
    
    matrix_num = len(matrix_list)
    output = np.zeros((matrix_num, rho_num, theta_num))
    
    segment_size = int(rho_num / n_segments)
    
    for index in range(n_segments):
        
        print(index)
        
        start = index * segment_size
        end = (index + 1) * segment_size
        
        rho_segment = rho_grid[start:end]
        
        grid_index = np.zeros((segment_size * theta_num, 3))
        for i in range(segment_size):
            for j in range(theta_num):
                grid_index[i * theta_num + j, 0] = rho_segment[i]
                grid_index[i * theta_num + j, 1] = theta_pest[j]
        
        field = eq.compute(matrix_list, desc.grid.Grid(grid_index, coordinates='rvp'))
        
        for k in range(matrix_num):
            temp = np.array(field[matrix_list[k]])
            for i in range(segment_size):
                for j in range(theta_num):
                    output[k, start + i, j] = temp[i * theta_num + j]
    
    path = outputPath + '/plot2D.mat'
    savemat(path,
            {'qplot': output[0],
             'Rplot': output[1],
             'Zplot': output[2],
             'rhoplot': output[3]})

### 第三步：读取和检查平衡

运行 `3.1代码块` 读取 `inputPath` 指定的平衡文件。

查看安全因子剖面、压强剖面、磁面形状、归一化力误差和磁轴物理量。

In [ ]:
################## 3.1代码块 ##################

eq = desc.io.load(inputPath)

profileGrid = desc.grid.LinearGrid(L=200)
profileData = eq.compute(["q", "p"], profileGrid)

plt.figure()
plt.plot(profileData["q"])
plt.title("Safety factor q")
plt.xlabel("rho")
plt.ylabel("q")

plt.figure()
plt.plot(profileData["p"])
plt.title("Pressure")
plt.xlabel("rho")
plt.ylabel("p")

desc.plotting.plot_surfaces(eq, 10, 36)
desc.plotting.plot_section(eq, "|F|_normalized", log=True)

u0 = 4 * np.pi * 1.0e-7
axisGrid = desc.grid.Grid(np.array([0, 0, 0]))
axisData = eq.compute(["|B|", "p", "R", "Z"], axisGrid)

B0 = axisData["|B|"][0]
p0 = axisData["p"][0]
R0 = axisData["R"][0]
Z0 = axisData["Z"][0]
beta0 = p0 / (B0 * B0 / 2 / u0)

print("B0    =", B0)
print("R0    =", R0)
print("Z0    =", Z0)
print("beta0 =", beta0)

### 第四步：输出平衡

本步骤将平衡输出为三个 MATLAB 文件：`standard2D.mat`、`refined2D.mat` 和 `plot2D.mat`。

先运行 `4.1代码块` 设置标准输出网格，再运行 `4.2代码块` 生成 `rho` 和 `theta_PEST` 网格。随后运行 `4.3代码块` 输出主平衡文件。

运行 `4.4代码块`得到加密网格用于插值。运行 `4.5代码块`得到高分辨率画图网格。

`4.1代码块` 设置标准输出网格：

1. `rhoMin`：输出区域的最小归一化径向坐标。
2. `rhoMax`：输出区域的最大归一化径向坐标。
3. `gridRho`：标准输出的径向网格数。
4. `gridTheta`：标准输出的 `theta_PEST` 网格数。
5. `ifStaggered`：是否使用交错 `theta_PEST` 网格；`0` 表示否，`1` 表示是。

In [ ]:
################## 4.1代码块 ##################

rhoMin = 0.08
rhoMax = 0.90
gridRho = 256
gridTheta = 32
ifStaggered = 0

`4.2代码块` 根据 `4.1代码块` 的网格设置，生成 `rho` 和 `theta_PEST` 网格。

不再构造 `theta_PEST -> theta_DESC` 插值，`outputStandard2D` 将直接使用 `rvp` 网格。

In [ ]:
################## 4.2代码块 ##################

rhoGrid, thetaPEST = getRhoThetaPEST(ifStaggered,gridRho,rhoMin,rhoMax,gridTheta)

`4.3代码块` 输出 `standard2D.mat`。这是主二维平衡文件，包含磁场、几何、度规、PEST 坐标量和剖面量。

`standardSegments` 是径向分段数，用于降低单次 `eq.compute` 的内存占用。`gridRho` 应能被 `standardSegments` 整除。

In [ ]:
################## 4.3代码块 ##################

start = time.perf_counter()

standardSegments = 8
outputStandard2D(ifStaggered, rhoGrid, thetaPEST, standardSegments)

end = time.perf_counter()
elapsed_ms = (end - start) * 1000
print(f"耗时：{elapsed_ms:.3f} ms")

`4.4代码块` 输出 `refined2D.mat`，用于后续插值。

1. `refinedRho0`：加密网格的最小径向坐标，小于 `rhoMin`。
2. `refinedRho1`：加密网格的最大径向坐标，大于 `rhoMax`。
3. `refinedRhoNum`：加密网格的径向网格数，大于 `gridRho`。
4. `refinedThetaNum`：加密网格的极向网格数，大于 `gridTheta`。
5. `refinedSegments`：径向分段数，用于降低内存占用；`refinedRhoNum` 应能被 `refinedSegments` 整除。

In [ ]:
################## 4.4代码块 ##################

refinedRho0 = 0.05
refinedRho1 = 0.95
refinedRhoNum = 2048
refinedThetaNum = 1024
refinedSegments = 8

start = time.perf_counter()

outputRefined2D(refinedRho0, refinedRho1, refinedRhoNum, refinedThetaNum, refinedSegments)

end = time.perf_counter()
elapsed_ms = (end - start) * 1000
print(f"耗时：{elapsed_ms:.3f} ms")

`4.5代码块` 输出 `plot2D.mat`，用于高分辨率绘图。

1. `plotRhoNum`：画图网格的径向网格数。
2. `plotThetaNum`：画图网格的极向网格数。
3. `plotSegments`：径向分段数，用于降低内存占用；`plotRhoNum` 应能被 `plotSegments` 整除。

In [ ]:
################## 4.5代码块 ##################

plotRhoNum = 1024
plotThetaNum = 1024
plotSegments = 8

start = time.perf_counter()

rhoGrid, thetaPEST = getRhoThetaPEST(0,plotRhoNum,rhoMin,rhoMax,plotThetaNum)
outputPlot2D(rhoGrid, thetaPEST, plotSegments)

end = time.perf_counter()
elapsed_ms = (end - start) * 1000
print(f"耗时：{elapsed_ms:.3f} ms")